In [ ]:
%pip -q install -U huggingface_hub

import os, re, json, hashlib, zipfile, stat, shutil, tempfile
from pathlib import Path, PurePosixPath
from huggingface_hub import HfApi, hf_hub_download
from google.colab import userdata

REPO_ID = 'Zzwahwahz/conquer-mobile-source-assets'
REPO_TYPE = 'dataset'
EXPECTED = {
    'ConquerOnline.zip': {'size': 461117324, 'sha256': 'b1c6958109a07c237631943fc0442d3161563d89e1020b143d95437ba9a6fdb0'},
    'Conquer-Mobile-Assets.-mobile-source-files.zip': {'size': 250389993, 'sha256': '5acb29a128f4560bd84a956a628411f12ee713ceb5e2ecb935f65d31ba37d4e2'},
}
BASE = Path('/content/conquer_mobile_work')
DOWNLOADS = BASE / 'downloads'
EXTRACTED = BASE / 'extracted'
REPORT_PATH = BASE / 'extraction-report.json'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    count, total_bytes = 0, 0
    with zipfile.ZipFile(zip_path) as zf:
        bad = zf.testzip()
        if bad:
            raise RuntimeError(f'ZIP CRC test failed at: {bad}')
        for info in zf.infolist():
            raw = info.filename.replace('\\', '/')
            member = PurePosixPath(raw)
            if member.is_absolute() or '..' in member.parts or not member.parts:
                raise RuntimeError(f'Unsafe ZIP path rejected: {info.filename}')
            mode = (info.external_attr >> 16) & 0xFFFF
            if stat.S_ISLNK(mode):
                raise RuntimeError(f'Symlink rejected: {info.filename}')
            target = (destination / Path(*member.parts)).resolve()
            if destination not in target.parents and target != destination:
                raise RuntimeError(f'ZIP path escapes destination: {info.filename}')
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info, 'r') as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
            count += 1
            total_bytes += info.file_size
    return count, total_bytes

print('Reading HF_TOKEN from Colab Secrets...')
token = userdata.get('HF_TOKEN')
if not token:
    raise RuntimeError('HF_TOKEN is missing. Add it in Colab Secrets and enable notebook access.')
api = HfApi(token=token)
api.repo_info(repo_id=REPO_ID, repo_type=REPO_TYPE)
print('Authenticated. Checking the source ZIP files in the private dataset...')
repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type=REPO_TYPE)
by_name = {Path(p).name: p for p in repo_files}
missing = [name for name in EXPECTED if name not in by_name]
if missing:
    raise FileNotFoundError('These ZIP files are not in the HF repository yet: ' + ', '.join(missing) + '. Upload both original ZIPs to the repository root, then rerun this cell.')

DOWNLOADS.mkdir(parents=True, exist_ok=True)
EXTRACTED.mkdir(parents=True, exist_ok=True)
report = {'repo_id': REPO_ID, 'repo_type': REPO_TYPE, 'files': {}, 'extracted_root': 'extracted/'}
for name, expected in EXPECTED.items():
    print(f'\\nDownloading {name} ...')
    local_zip = Path(hf_hub_download(repo_id=REPO_ID, filename=by_name[name], repo_type=REPO_TYPE, token=token, force_download=True))
    actual_size = local_zip.stat().st_size
    if actual_size != expected['size']:
        raise RuntimeError(f'{name}: size mismatch; expected {expected["size"]}, got {actual_size}')
    digest = sha256_file(local_zip)
    if digest.lower() != expected['sha256']:
        raise RuntimeError(f'{name}: SHA-256 mismatch; got {digest}')
    target_dir = EXTRACTED / Path(name).stem
    target_dir.mkdir(parents=True, exist_ok=True)
    file_count, uncompressed_bytes = safe_extract(local_zip, target_dir)
    report['files'][name] = {'source_path': by_name[name], 'size_bytes': actual_size, 'sha256': digest, 'extracted_to': str(target_dir.relative_to(BASE)), 'extracted_files': file_count, 'uncompressed_bytes': uncompressed_bytes}
    print(f'OK: SHA-256 verified; extracted {file_count:,} files ({uncompressed_bytes:,} bytes).')

all_files = [p for p in EXTRACTED.rglob('*') if p.is_file()]
if not all_files:
    raise RuntimeError('Extraction produced no files; refusing to upload an empty result.')
report['total_extracted_files'] = len(all_files)
report['total_extracted_bytes'] = sum(p.stat().st_size for p in all_files)
report['status'] = 'extraction_verified_upload_pending'
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'\\nUploading {len(all_files):,} extracted files to {REPO_ID} under extracted/...')
api.upload_folder(folder_path=str(EXTRACTED), repo_id=REPO_ID, repo_type=REPO_TYPE, path_in_repo='extracted', commit_message='Add verified extracted Conquer Mobile source assets')
api.upload_file(path_or_fileobj=str(REPORT_PATH), path_in_repo='extraction-report.json', repo_id=REPO_ID, repo_type=REPO_TYPE, commit_message='Add Conquer Mobile extraction verification report')

print('Verifying uploaded repository paths...')
final_files = api.list_repo_files(repo_id=REPO_ID, repo_type=REPO_TYPE)
uploaded = [p for p in final_files if p.startswith('extracted/') and not p.endswith('/')]
missing_after = []
for local in all_files:
    rel = local.relative_to(EXTRACTED).as_posix()
    remote = 'extracted/' + rel
    if remote not in final_files:
        missing_after.append(remote)
if missing_after:
    raise RuntimeError(f'Upload verification failed: {len(missing_after)} files missing. First examples: {missing_after[:10]}')
if 'extraction-report.json' not in final_files:
    raise RuntimeError('Report upload verification failed.')
print('\\nSUCCESS: both ZIPs were checksum-verified, extracted, uploaded, and checked against the Hub file listing.')
print(f'Extracted files verified on Hub: {len(uploaded):,}')
print('Report: https://huggingface.co/datasets/' + REPO_ID + '/blob/main/extraction-report.json')
